In [ ]:

from pyspark.sql import functions as F

from atlas.common.config.loader import get_settings
from atlas.common.paths.loader import get_paths
from atlas.common.spark.session import get_spark_session

settings = get_settings("configs/base.yaml", "configs/local.yaml", "pyproject.toml")

In [ ]:
spark = get_spark_session(settings.spark, settings.storage, settings.application.name)

In [ ]:
paths = get_paths(settings)

In [ ]:
bronze_customer_path = paths.bronze_path("customer/cdc/customers/job")

In [ ]:
customer_bronze_data = spark.read.format("parquet").load(bronze_customer_path)

In [ ]:

customer_bronze_data.select(F.col("raw_key"), F.col("raw_value")).show(vertical=True, truncate=False, n=1)

In [ ]:
from pyspark.sql.types import LongType, StringType, StructField, StructType

customer_record_schema = StructType([
    StructField("customer_id", LongType(), False),
    StructField("first_name", StringType(), False),
    StructField("last_name", StringType(), False),
    StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True),
    StructField("date_of_birth", LongType(), True),
    StructField("status", StringType(), False),
    StructField("segment", StringType(), False),
    StructField("created_at", StringType(), False),
    StructField("updated_at", StringType(), False),
])

In [ ]:
customer_source_schema = StructType([
    StructField("version", StringType(), False),
    StructField("connector", StringType(), False),
    StructField("name", StringType(), False),
    StructField("ts_ms", LongType(), False),
    StructField("snapshot", StringType(), True),
    StructField("db", StringType(), False),
    StructField("sequence", StringType(), True),
    StructField("ts_us", LongType(), True),
    StructField("ts_ns", LongType(), True),
    StructField("schema", StringType(), False),
    StructField("table", StringType(), False),
    StructField("txId", LongType(), True),
    StructField("lsn", LongType(), True),
    StructField("xmin", LongType(), True),
])

In [ ]:
customer_payload_schema = StructType([
    StructField("before", customer_record_schema, True),
    StructField("after", customer_record_schema, True),
    StructField("source", customer_source_schema, False),
    StructField("op", StringType(), False),
    StructField("ts_ms", LongType(), False),
    StructField("ts_us", LongType(), True),
    StructField("ts_ns", LongType(), True),
])

In [ ]:
customer_debezium_schema = StructType([
    StructField("payload", customer_payload_schema, True),
])

In [ ]:
customer_parsed_data = customer_bronze_data.withColumn("debezium", F.from_json(F.col("raw_value")
                                                                               , customer_debezium_schema))

In [ ]:
customer_parsed_data.select(
    "debezium.payload.before",
    "debezium.payload.after",
    "debezium.payload.op",
    "debezium.payload.source.lsn",
    "kafka_partition",
    "kafka_offset",
).show(1, truncate=False, vertical=True)

In [ ]:
customer_cdc_data = customer_parsed_data.withColumn("customer",
                                                    F.when(
                                                        F.col("debezium.payload.op") == "d",
                                                        F.col("debezium.payload.before"),

                                                    ).otherwise(
                                                        F.col("debezium.payload.after")
                                                    )
                                                    )

In [ ]:
customer_cdc_data_normalized = customer_cdc_data.select(
    F.col("customer.customer_id").alias("customer_id"),
    F.col("customer.first_name").alias("first_name"),
    F.col("customer.last_name").alias("last_name"),
    F.col("customer.email").alias("email"),
    F.col("customer.phone_number").alias("phone_number"),
    F.date_add(
        F.lit("1970-01-01").cast("date"),
        F.col("customer.date_of_birth").cast("int")
    ).alias("date_of_birth"),
    F.col("customer.status").alias("status"),
    F.col("customer.segment").alias("segment"),
    F.try_to_timestamp(F.col("customer.created_at")).alias("created_at"),
    F.try_to_timestamp(F.col("customer.updated_at")).alias("updated_at"),
    F.timestamp_millis(F.col("debezium.payload.ts_ms")).alias("cdc_timestamp"),
    F.timestamp_millis(F.col("debezium.payload.source.ts_ms")).alias("source_timestamp"),
    F.col("debezium.payload.op").alias("cdc_operation"),
    F.col("debezium.payload.source.lsn").alias("source_lsn"),
    F.col("debezium.payload.source.txId").alias("source_tx_id"),
    F.col("kafka_topic").alias("kafka_topic"),
    F.col("kafka_partition").alias("kafka_partition"),
    F.col("kafka_offset").alias("kafka_offset"),
    F.col("kafka_timestamp").alias("kafka_timestamp"),
    F.col("is_tombstone").alias("is_tombstone"),
    F.col("ingested_at").alias("ingested_at"),
)

In [ ]:
customer_non_tombstone_data = customer_cdc_data_normalized.filter(
    ~F.col("is_tombstone")
)

In [ ]:
customer_filter_condition = (
    F.array(
        F.when(F.col("customer_id").isNull(), F.lit("MISSING_CUSTOMER_ID")),
        F.when((F.col("first_name").isNull()| (F.trim(F.col("first_name")) == "")), F.lit("MISSING_FIRST_NAME")),
        F.when((F.col("last_name").isNull() |(F.trim(F.col("last_name")) == "")), F.lit("MISSING_LAST_NAME")),
        F.when((F.col("email").isNull() & F.col("phone_number").isNull()), F.lit("MISSING_CONTACT_INFO")),
        F.when(F.col("date_of_birth") > F.current_date(), F.lit("FUTURE_DATE_OF_BIRTH")),
        F.when(~F.col("status").isin(["ACTIVE","INACTIVE","SUSPENDED"]), F.lit("INVALID_STATUS")),
        F.when(~F.col("segment").isin(["STANDARD","GOLD","PREMIUM"]), F.lit("INVALID_SEGMENT"))
))

In [ ]:
customer_filtered_data = customer_non_tombstone_data.withColumn("dq_errors", F.array_compact(customer_filter_condition))

In [ ]:
customer_filtered_data.select(["customer_id","first_name","last_name","email","phone_number"
                                  ,"cdc_operation","is_tombstone","dq_errors" ]).show(truncate=False)

In [ ]:
(customer_filtered_data.select(["customer_id","first_name","last_name","email",
                               "phone_number","cdc_operation","is_tombstone","dq_errors" ])
 .show(truncate=False))

In [ ]:
customer_valid_data = customer_filtered_data.filter(F.size(F.col("dq_errors")) ==0)
customer_quarantine_data = customer_filtered_data.filter(F.size(F.col("dq_errors")) >0)

In [ ]:
customer_quarantine_data = (customer_quarantine_data.withColumn("dq_error_count", F.size(F.col("dq_errors")))
                            .withColumn("quarantined_at", F.current_timestamp()))

In [ ]:
# Persist DQ-invalid customer records for investigation
silver_customer_quarantine_path = paths.silver_path(
    "customer/cdc/customers/quarantine/notebook"
)


In [ ]:
customer_quarantine_data.show(truncate=False, vertical=True)

In [ ]:
customer_valid_data = customer_valid_data.drop("dq_errors")

In [ ]:
customer_deduplicated_data = customer_valid_data.drop_duplicates(["kafka_topic","kafka_partition", "kafka_offset"])

In [ ]:
customer_deduplicated_data.select(
    "customer_id",
    "cdc_operation",
    "source_tx_id",
    "source_lsn",
    "kafka_partition",
    "kafka_offset",
).orderBy(
    "kafka_partition",
    "kafka_offset",
).show(truncate=False)


In [ ]:
customer_deduplicated_data.count()

In [ ]:
customer_incoming_ambiguous_keys = (customer_deduplicated_data.groupBy(['customer_id', 'source_lsn'])
                                    .agg(F.countDistinct("kafka_partition").alias("distinct_partition") )
                                    .filter(F.col("distinct_partition")>=2))

In [ ]:
customer_incoming_ambiguous_events = customer_deduplicated_data.join(customer_incoming_ambiguous_keys,
                                                                     ["customer_id","source_lsn"],"left_semi")
customer_incoming_ambiguous_events = (
    customer_incoming_ambiguous_events
    .withColumn("cdc_status", F.lit("AMBIGUOUS_ORDERING"))
    .withColumn("persisted_source_lsn", F.lit(None).cast("long"))
    .withColumn("persisted_kafka_partition", F.lit(None).cast("int"))
    .withColumn("persisted_kafka_offset", F.lit(None).cast("long"))
    .withColumn("rejected_at", F.current_timestamp())
)

In [ ]:
customer_incoming_orderable_data = customer_deduplicated_data.join(
    customer_incoming_ambiguous_keys,
    ["customer_id", "source_lsn"],
    "left_anti"
)

In [ ]:
# S3 - CDC ordering, replay protection and persistent history

from delta.tables import DeltaTable
from pyspark.sql import Window


In [ ]:
# Persistent Delta table containing observed CDC event history

silver_customer_history_path = paths.silver_path(
    "customer/cdc/customers/cdc_history/notebook"
)
# Check history before processing this run

customer_history_exists = DeltaTable.isDeltaTable(
    spark,
    silver_customer_history_path
)

In [ ]:
if customer_history_exists:

    # Read PREVIOUS accepted history
    customer_cdc_history_table = DeltaTable.forPath(
        spark,
        silver_customer_history_path
    )

    customer_cdc_history_data = customer_cdc_history_table.toDF()

    # Latest accepted event for each customer
    customer_cdc_latest_window = Window.partitionBy(
        "customer_id"
    ).orderBy(
        F.col("source_lsn").desc(),
        F.col("kafka_offset").desc()
    )

    customer_cdc_latest = (
        customer_cdc_history_data
        .withColumn(
            "row_number",
            F.row_number().over(customer_cdc_latest_window)
        )
        .filter(F.col("row_number") == 1)
        .drop("row_number")
    )

    # Compare this run's incoming events against PREVIOUS history
    customer_cdc_comparison = (
       customer_incoming_orderable_data.alias("s")
        .join(
            customer_cdc_latest.alias("t"),
            F.col("s.customer_id") == F.col("t.customer_id"),
            "left"
        )
    )

    # Classify source ordering
    customer_cdc_classified = customer_cdc_comparison.withColumn(
        "cdc_status",

        F.when(
            F.col("t.customer_id").isNull(),
            F.lit("NEW")
        )
        .when(
            F.col("s.source_lsn") > F.col("t.source_lsn"),
            F.lit("NEWER")
        )
        .when(
            F.col("s.source_lsn") < F.col("t.source_lsn"),
            F.lit("STALE")
        )
        .when(
            F.col("s.kafka_partition") != F.col("t.kafka_partition"),
            F.lit("AMBIGUOUS_ORDERING")
        )
        .when(
            F.col("s.kafka_offset") > F.col("t.kafka_offset"),
            F.lit("NEWER")
        )
        .otherwise(
            F.lit("STALE")
        )
    )

    # Only accepted incoming events
    customer_cdc_accepted_events = (
        customer_cdc_classified
        .filter(F.col("cdc_status").isin("NEW", "NEWER"))
        .select("s.*")
    )

    # Keep these separately for later monitoring/quarantine
    customer_cdc_rejected_events = (
    customer_cdc_classified
    .filter(
        F.col("cdc_status").isin(
            "STALE",
            "AMBIGUOUS_ORDERING"
        )
    )
    .select(
        # Keep incoming event once
        "s.*",
        # Why we rejected it
        "cdc_status",
        # Previous accepted position it conflicted with
        F.col("t.source_lsn").alias("persisted_source_lsn"),
        F.col("t.kafka_partition").alias(
            "persisted_kafka_partition"
        ),
        F.col("t.kafka_offset").alias(
            "persisted_kafka_offset"
        )
    )
    .withColumn(
        "rejected_at",
        F.current_timestamp()
    )
)
else:
    # FIRST RUN:
    # There is no previous history to compare against.
    customer_cdc_accepted_events = customer_incoming_orderable_data
    customer_cdc_rejected_events = customer_incoming_ambiguous_events

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F


def merge_cdc_events(
    spark: SparkSession,
    data: DataFrame,
    target_path: str,
) -> None:
    """Persist CDC events idempotently using Kafka record identity."""

    if not DeltaTable.isDeltaTable(spark, target_path):
        (
            data.write
            .format("delta")
            .save(target_path)
        )
        return

    target_table = DeltaTable.forPath(
        spark,
        target_path
    )

    event_identity_condition = (
        (F.col("t.kafka_topic") == F.col("s.kafka_topic"))
        & (F.col("t.kafka_partition") == F.col("s.kafka_partition"))
        & (F.col("t.kafka_offset") == F.col("s.kafka_offset"))
    )

    (
        target_table.alias("t")
        .merge(
            data.alias("s"),
            event_identity_condition
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [ ]:
silver_customer_cdc_rejected_path = paths.silver_path(
    "customer/cdc/customers/rejected/notebook"
    )
silver_customer_quarantine_path = paths.silver_path(
    "customer/cdc/customers/quarantine/notebook"
)

In [ ]:
# DQ quarantine
merge_cdc_events(
    spark,
    customer_quarantine_data,
    silver_customer_quarantine_path,
)

# Accepted CDC history
merge_cdc_events(
    spark,
    customer_cdc_accepted_events,
    silver_customer_history_path,
)


# Persist CDC ordering rejections
if customer_cdc_rejected_events is not None:
    customer_cdc_rejected_events = (
    customer_cdc_rejected_events
    .unionByName(customer_incoming_ambiguous_events)
)
    merge_cdc_events(
        spark,
        customer_cdc_rejected_events,
        silver_customer_cdc_rejected_path,
    )

In [ ]:
print(customer_cdc_rejected_events)

In [ ]:
customer_latest_accepted_window = (Window.partitionBy("customer_id")
                                   .orderBy(F.col("source_lsn").desc(), F.col("kafka_offset").desc()))

In [ ]:
from datetime import date

s4_test_data = [
    # Existing customer, two updates in same micro-batch
    (1, "Sailesh", "Naidu", "sailesh@test.com", "9999999999",
     date(1995, 1, 1), "SUSPENDED", "GOLD",
     "u", 200, "atlas.customer.public.customers", 0, 20),

    (1, "Sailesh", "Naidu", "sailesh@test.com", "9999999999",
     date(1995, 1, 1), "ACTIVE", "GOLD",
     "u", 220, "atlas.customer.public.customers", 0, 21),

    # Existing customer gets deleted
    (2, "Aarav", "Sharma", "aarav@test.com", "8888888888",
     date(1994, 1, 1), "ACTIVE", "STANDARD",
     "d", 230, "atlas.customer.public.customers", 0, 22),

    # New customer
    (4, "Maya", "Patel", "maya@test.com", "7777777777",
     date(1996, 1, 1), "ACTIVE", "PREMIUM",
     "c", 240, "atlas.customer.public.customers", 0, 23),
]

s4_test_schema = """
customer_id LONG,
first_name STRING,
last_name STRING,
email STRING,
phone_number STRING,
date_of_birth DATE,
status STRING,
segment STRING,
cdc_operation STRING,
source_lsn LONG,
kafka_topic STRING,
kafka_partition INT,
kafka_offset LONG
"""

customer_cdc_accepted_events_test = spark.createDataFrame(
    s4_test_data,
    schema=s4_test_schema,
)

In [ ]:
customer_cdc_accepted_events_test.orderBy(
    "customer_id",
    "source_lsn"
).show(truncate=False)

In [ ]:
customer_latest_accepted_records_test = (customer_cdc_accepted_events_test
                                         .withColumn("rn", F.row_number().over(customer_latest_accepted_window))
                                         .filter(F.col("rn") ==1)
                                         .drop(F.col("rn")))

In [ ]:
customer_latest_accepted_records_test.select(
    "customer_id",
    "status",
    "cdc_operation",
    "source_lsn",
    "kafka_offset",
).orderBy("customer_id").show()

In [ ]:
silver_customer_current_path = paths.silver_path(
    "customer/cdc/customers/canonical/notebook"
)

In [ ]:
if not DeltaTable.isDeltaTable(spark, silver_customer_current_path):
    customer_initial_current_records = (
        customer_latest_accepted_records_test
        .filter(F.col("cdc_operation") != "d")
    )

    (
        customer_initial_current_records.write
        .format("delta")
        .save(silver_customer_current_path)
    )


In [ ]:
customer_current = (
    spark.read
    .format("delta")
    .load(silver_customer_current_path)
)

customer_current.select(
    "customer_id",
    "status",
    "cdc_operation",
    "source_lsn",
).orderBy("customer_id").show()

In [ ]:
s4_merge_test_data = [
    # Existing customer 1 gets updated
    (1, "Sailesh", "Naidu", "sailesh.new@test.com", "9999999999",
     date(1995, 1, 1), "SUSPENDED", "PREMIUM",
     "u", 300, "atlas.customer.public.customers", 0, 30),

    # Existing customer 4 gets deleted
    (4, "Maya", "Patel", "maya@test.com", "7777777777",
     date(1996, 1, 1), "ACTIVE", "PREMIUM",
     "d", 310, "atlas.customer.public.customers", 0, 31),

    # New customer 5 gets created
    (5, "Rohan", "Mehta", "rohan@test.com", "6666666666",
     date(1993, 1, 1), "ACTIVE", "STANDARD",
     "c", 320, "atlas.customer.public.customers", 0, 32),

    # Delete for customer that doesn't exist in canonical
    (6, "Neha", "Kapoor", "neha@test.com", "5555555555",
     date(1997, 1, 1), "ACTIVE", "GOLD",
     "d", 330, "atlas.customer.public.customers", 0, 33),
]

customer_cdc_accepted_events_merge_test = spark.createDataFrame(
    s4_merge_test_data,
    schema=s4_test_schema,
)

In [ ]:
customer_cdc_accepted_events_merge_test.orderBy(
    "customer_id",
    "source_lsn"
).show(truncate=False)

In [ ]:
customer_latest_accepted_records_test.select(
    "customer_id",
    "status",
    "cdc_operation",
    "source_lsn",
    "kafka_offset",
).orderBy("customer_id").show()

In [ ]:
customer_current_table = DeltaTable.forPath(
    spark,
    silver_customer_current_path
)

(
    customer_current_table.alias("t")
    .merge(
        customer_cdc_accepted_events_merge_test.alias("s"),
        F.col("t.customer_id") == F.col("s.customer_id")
    )
    .whenMatchedUpdateAll(
        condition="s.cdc_operation in ('c','u','r')"
    )
    .whenMatchedDelete(
        condition="s.cdc_operation = 'd'"
    )
    .whenNotMatchedInsertAll(
         condition="s.cdc_operation in ('c','u','r')"
    )
   .execute()
)

In [ ]:
customer_current_table.toDF().show()

In [ ]:
from atlas.common.paths.get_cdc_paths import get_bronze_paths, get_silver_paths

bronze_customer_path, _ = get_bronze_paths(
    settings,
    "customer",
    "customers",
)

bronze_test = (
    spark.read
    .format("parquet")
    .load(bronze_customer_path)
)

bronze_test.select(
    "kafka_partition",
    "kafka_offset",
    "is_tombstone",
).orderBy(
    "kafka_partition",
    "kafka_offset",
).show(100, truncate=False)

In [ ]:
(DeltaTable.forPath(spark, get_silver_paths(settings,"customer", "customers","canonical")).toDF()
 .filter(F.col("customer_id")==6).show(vertical = True, truncate=False))

In [ ]:
from atlas.common.paths.get_cdc_paths import get_reconciliation_paths

run_summary_path = get_reconciliation_paths(settings, "customer", "customers", "run_summary", )

In [ ]:
DeltaTable.forPath(spark, run_summary_path).toDF().printSchema()

In [ ]:
previous_snapshot_as_of =(DeltaTable.forPath(spark, run_summary_path)
    .toDF()
    .agg(F.max("snapshot_as_of").alias("snapshot_as_of"))
    .select(
        F.date_format(
            F.col("snapshot_as_of"),
            "yyyy-MM-dd HH:mm:ss"
        ).alias("snapshot_as_of")
    )
    .first()["snapshot_as_of"]
)


print(previous_snapshot_as_of)

In [ ]:
latest_snapshot_df = (
    DeltaTable.forPath(spark, run_summary_path)
    .toDF()
    .agg(F.max("snapshot_as_of").alias("snapshot_as_of"))
)
latest_snapshot_time = latest_snapshot_df.select("snapshot_as_of")
print(latest_snapshot_time)

In [ ]:
expected_state_metadata_path = get_reconciliation_paths(
    settings,
    "customer",
    "customers",
    "expected_state_metadata",
)

expected_state_metadata_df = (
    DeltaTable
    .forPath(spark, expected_state_metadata_path)
    .toDF()
)

expected_state_metadata_df.show(truncate=False, vertical=True)

In [ ]:
latest_state_as_of_df = (
    expected_state_metadata_df
    .agg(F.max("state_as_of").alias("state_as_of"))
)

In [ ]:
silver_customer_history_path = get_silver_paths(
    settings,
    "customer",
    "customers",
    "cdc_history",
)

customer_cdc_history = (
    DeltaTable
    .forPath(spark, silver_customer_history_path)
    .toDF()
)

In [ ]:
customer_cdc_history.select(
    "customer_id",
    "cdc_operation",
    "cdc_timestamp",
    "source_timestamp",
    "source_lsn",
    "kafka_partition",
    "kafka_offset",
).orderBy(
    "source_timestamp",
    "source_lsn",
).show(100, truncate=False)

In [ ]:
current_state_as_of = "2026-09-14 03:51:00"

In [ ]:
incremental_cdc_changes = (
    customer_cdc_history
    .crossJoin(
        latest_state_as_of_df
        .withColumnRenamed(
            "state_as_of",
            "previous_state_as_of",
        )
    )
    .filter(
        (F.col("source_timestamp") > F.col("previous_state_as_of"))
        &
        (
            F.col("source_timestamp")
            <= F.to_timestamp(F.lit(current_state_as_of))
        )
    )
)

In [ ]:
incremental_cdc_changes.select(
    "customer_id",
    "cdc_operation",
    "source_timestamp",
    "source_lsn",
    "kafka_partition",
    "kafka_offset",
).orderBy(
    "customer_id",
    "source_lsn",
).show(truncate=False)

In [ ]:
incremental_latest_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(
        F.col("source_lsn").desc(),
        F.col("kafka_offset").desc(),
    )
)

In [ ]:
latest_incremental_customer_changes = (
    incremental_cdc_changes
    .withColumn(
        "rn",
        F.row_number().over(incremental_latest_window),
    )
    .filter(F.col("rn") == 1)
    .drop("rn", "previous_state_as_of")
)

In [ ]:
latest_incremental_customer_changes.select(
    "customer_id",
    "cdc_operation",
    "source_timestamp",
    "source_lsn",
    "kafka_offset",
).orderBy(
    "customer_id",
).show(truncate=False)

In [ ]:
expected_state_path = get_reconciliation_paths(
    settings,
    "customer",
    "customers",
    "expected_state",
)

expected_state_table = DeltaTable.forPath(
    spark,
    expected_state_path,
)

In [ ]:
expected_state_table.toDF().select(
    "customer_id",
    "status",
    "segment",
    "cdc_operation",
    "source_lsn",
    "source_timestamp",
).orderBy("customer_id").show(truncate=False)

In [ ]:
latest_incremental_customer_changes_for_merge = (
    latest_incremental_customer_changes
    .withColumn(
        "state_updated_as_of",
        F.to_timestamp(F.lit(current_state_as_of)),
    )
)

In [ ]:
merge_condition = "target.customer_id = source.customer_id"
expected_state_after_merge =(
    expected_state_table.alias("target")
    .merge(
        latest_incremental_customer_changes_for_merge.alias("source"),
        merge_condition,
    )
    .whenMatchedUpdateAll(
        condition="source.cdc_operation IN ('c', 'u', 'r')"
    )
    .whenMatchedDelete(
        condition="source.cdc_operation = 'd'"
    )
    .whenNotMatchedInsertAll(
        condition="source.cdc_operation IN ('c', 'u', 'r')"
    )
    .execute()
)

In [ ]:
expected_state_after_merge.show()

In [ ]:
expected_state_after_merge.select(
    "customer_id",
    "status",
    "segment",
    "cdc_operation",
    "source_lsn",
    "source_timestamp",
).orderBy("customer_id").show(truncate=False)

In [ ]:
(DeltaTable.forPath(spark,
"/Users/saileshpola/Desktop/AtlasProject/data/lakehouse/ops/reconciliation/customer/customer_addresses/run_summary")
 .toDF().show(vertical = True))

In [ ]:
(DeltaTable.forPath(spark,
        "/Users/saileshpola/Desktop/AtlasProject/data/lakehouse/ops/reconciliation/customer/customer_addresses/expected_state")
 .toDF().show(vertical = True))

In [94]:
(DeltaTable.forPath(spark,
        "/Users/saileshpola/Desktop/AtlasProject/data/lakehouse/ops/reconciliation/customer/customer_consents/run_summary")
 .toDF().show())

+------------------+-------------+---------------+--------------------+-------------------------+-----------------------+-----------------+--------------+---------------------+-----------------+-------------------+--------------------+
|snapshot_row_count|cdc_row_count|exception_count|missing_in_cdc_count|missing_in_snapshot_count|checksum_mismatch_count|matched_row_count|overall_status|reconciliation_run_id|      entity_name|     snapshot_as_of|       reconciled_at|
+------------------+-------------+---------------+--------------------+-------------------------+-----------------------+-----------------+--------------+---------------------+-----------------+-------------------+--------------------+
|                13|           13|              0|                   0|                        0|                      0|               13|       SUCCESS| 74551664-2158-46d...|customer_consents|2026-09-14 09:55:00|2026-09-14 10:19:...|
|                13|           13|              0|      

In [89]:
(DeltaTable.forPath(spark,
    "/Users/saileshpola/Desktop/AtlasProject/data/lakehouse/ops/reconciliation/customer/customer_consents/expected_state")
 .toDF().show(vertical = True))

-RECORD 0-----------------------------------
 consent_id          | 2                    
 source_lsn          | 27069024             
 customer_id         | 1                    
 consent_type        | EMAIL                
 granted             | true                 
 created_at          | 2026-09-14 09:45:... 
 updated_at          | 2026-09-14 09:45:... 
 cdc_timestamp       | 2026-09-14 09:45:... 
 source_timestamp    | 2026-09-14 09:45:... 
 cdc_operation       | c                    
 kafka_topic         | atlas.customer.pu... 
 kafka_partition     | 0                    
 kafka_offset        | 15                   
 kafka_timestamp     | 2026-09-14 09:45:... 
 is_tombstone        | false                
 ingested_at         | 2026-09-14 09:46:... 
 state_updated_as_of | 2026-09-14 15:20:00  
-RECORD 1-----------------------------------
 consent_id          | 3                    
 source_lsn          | 27069448             
 customer_id         | 1                    
 consent_t

In [98]:
(DeltaTable.forPath(spark,
    "/Users/saileshpola/Desktop/AtlasProject/data/lakehouse/ops/reconciliation/customer/customer_consents/expected_state_metadata")
 .toDF().show())

26/09/14 15:50:43 WARN DeltaLog: Change in the table id detected while updating snapshot. 
Previous snapshot = Snapshot(path=file:/Users/saileshpola/Desktop/AtlasProject/data/lakehouse/ops/reconciliation/customer/customer_consents/expected_state_metadata/_delta_log, version=1, metadata=Metadata(4b22b675-1887-4811-a7c1-627ab03295d6,null,null,Format(parquet,Map()),{"type":"struct","fields":[{"name":"state_row_count","type":"long","nullable":true,"metadata":{}},{"name":"entity_name","type":"string","nullable":true,"metadata":{}},{"name":"previous_state_as_of","type":"timestamp","nullable":true,"metadata":{}},{"name":"state_as_of","type":"timestamp","nullable":true,"metadata":{}},{"name":"records_inserted","type":"long","nullable":true,"metadata":{}},{"name":"records_updated","type":"long","nullable":true,"metadata":{}},{"name":"records_deleted","type":"long","nullable":true,"metadata":{}},{"name":"processed_cdc_rows","type":"long","nullable":true,"metadata":{}},{"name":"reconciliation_run

+---------------+-----------------+--------------------+-------------------+----------------+---------------+---------------+------------------+---------------------+--------------------+
|state_row_count|      entity_name|previous_state_as_of|        state_as_of|records_inserted|records_updated|records_deleted|processed_cdc_rows|reconciliation_run_id|        processed_at|
+---------------+-----------------+--------------------+-------------------+----------------+---------------+---------------+------------------+---------------------+--------------------+
|             13|customer_consents| 2026-09-14 09:50:00|2026-09-14 09:55:00|               1|              2|              1|                 4| 74551664-2158-46d...|2026-09-14 10:18:...|
|             13|customer_consents|                NULL|2026-09-14 09:50:00|              13|              0|              0|                13| 56b26af9-7128-45d...|2026-09-14 10:17:...|
+---------------+-----------------+--------------------+----

26/09/14 20:34:01 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 514436 ms exceeds timeout 120000 ms
26/09/14 20:34:01 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/14 20:34:09 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [96]:
(DeltaTable.forPath(spark, "/Users/saileshpola/Desktop/AtlasProject/data/lakehouse/silver/customer/cdc/customer_consents/canonical/job")
 .toDF().show())

+----------+----------+-----------+------------+-------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+---------------+------------+--------------------+------------+--------------------+
|consent_id|source_lsn|customer_id|consent_type|granted|          created_at|          updated_at|       cdc_timestamp|    source_timestamp|cdc_operation|         kafka_topic|kafka_partition|kafka_offset|     kafka_timestamp|is_tombstone|         ingested_at|
+----------+----------+-----------+------------+-------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+---------------+------------+--------------------+------------+--------------------+
|         2|  27069024|          1|       EMAIL|   true|2026-09-14 09:45:...|2026-09-14 09:45:...|2026-09-14 09:45:...|2026-09-14 09:45:...|            c|atlas.customer.pu...|              0|          15|2026-09-14 09:45